# Image Classification - Lab

## Introduction

Now that you have a working knowledge of CNNs and have practiced implementing associated techniques in Keras, its time to put all of those skills together. In this lab, you'll work to complete a Kaggle competition on classifying dog breeds.

https://www.kaggle.com/c/dog-breed-identification

## Objectives

You will be able to:
* Independently design and build a CNN for image classifcation tasks
* Compare and apply multiple techniques for tuning a model including data augmentation and adapting pretrained models

## Download and Load the Data

Start by downloading the data locally and loading it into a Pandas DataFrame. Be forewarened that this dataset is fairly large and it is advisable to close other memory intensive applications.

The data can be found here:

https://www.kaggle.com/c/dog-breed-identification/data

We recommend downloading the data into this directory on your local computer. From there, be sure to uncompress the folder and subfolders.

In [1]:
#No code persay, but download and decompress the data.

## Preprocessing

Now that you've downloaded the data, its time to prepare it for some model building! You'll notice that the current structure provided is not the same as our lovely preprocessed folders that we've been providing you. Instead, you have one large training folder with images and a csv file with labels associated with each of these file types. 

Use this to create a directory substructure for a train-validation-test split as we have done previously. Also recall from our previous work that you'll also want to use one-hot encoding as we are now presented with a multi-class problem as opposed to simple binary classification.

In [1]:
#Your code here; open the labels.csv file stored in the zip file
import pandas as pd
import numpy as np
labels = pd.read_csv('labels.csv')
labels.head()

,id,breed
0,000bec180eb18c7604dcecc8fe0dba07,boston_bull
1,001513dfcb2ffafc82cccf4d8bbaba97,dingo
2,001cdf01b096e06d78e9e5112d419397,pekinese
3,00214f311d5d2247d5dfe4fe24b2303d,bluetick
4,0021f9ceb3235effd7fcde7f7538ed62,golden_retriever


We wish to create our standard directory structure:
* train
    * category1
    * category2
    * category3
    ...
* val
    * category1
    * category2
    * category3
    ...
* test 
    * category1
    * category2
    * category3
    ...  

In [6]:
#Your code here; transform the image files and then load them into Keras as tensors 
#(be sure to perform a train-val-test split)
train = pd.DataFrame()
test = pd.DataFrame()
val = pd.DataFrame()

In [7]:
for i in labels.breed.unique():
    breed_data = labels[labels['breed'] == i]
    size = breed_data.shape[0]
    val_data = breed_data.sample(int(round(size*.2,0)))
    val = val.append(val_data)
    
    breed_data = breed_data.drop(val_data.index)
    train = train.append(breed_data)

In [8]:
train.shape[0], test.shape[0], val.shape[0]

(8185, 0, 2037)

In [103]:
train.breed.value_counts()

scottish_deerhound                81
maltese_dog                       75
entlebucher                       74
afghan_hound                      74
bernese_mountain_dog              73
shih-tzu                          72
pomeranian                        71
great_pyrenees                    71
basenji                           70
samoyed                           70
airedale                          69
tibetan_terrier                   69
leonberg                          68
cairn                             68
beagle                            67
japanese_spaniel                  67
miniature_pinscher                66
blenheim_spaniel                  66
australian_terrier                66
irish_wolfhound                   65
lakeland_terrier                  63
saluki                            63
papillon                          62
norwegian_elkhound                61
whippet                           61
siberian_husky                    61
pug                               60
p

In [10]:
import os
o_dir = 'train/train/'
train_dir = 'data/train'
val_dir = 'data/val'
test_dir = 'data/test'
os.mkdir('data/')

In [11]:
import os
import shutil

try:
    os.mkdir(train_dir)
except:
    pass
for index, row in train.iterrows():
    id = row['id']
    breed = row['breed']
    try:
        os.mkdir(train_dir + '/' + breed)
    except:
        pass
    file = id + '.jpg'
    origin = os.path.join(o_dir + file)
    destination = os.path.join(train_dir + '/' + breed + '/' + file)
    shutil.copy(origin, destination)
#
try:
    os.mkdir(val_dir)  
except:
    pass
for index, row in val.iterrows():
    id = row['id']
    breed = row['breed']
    try:
        os.mkdir(val_dir + '/' + breed)
    except:
        pass
    file = id + '.jpg'
    origin = os.path.join(o_dir + file)
    destination = os.path.join(val_dir + '/' + breed + '/' + file)
    shutil.copy(origin, destination)

In [3]:
from keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(240, 240),
        batch_size=20,
        class_mode='categorical')

test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=(240, 240),
        batch_size=20,
        class_mode='categorical',
        shuffle=False)

validation_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=(240, 240),
        batch_size=20,
        class_mode='categorical')

Using TensorFlow backend.


Found 6547 images belonging to 120 classes.
Found 2037 images belonging to 120 classes.
Found 1638 images belonging to 120 classes.


In [26]:
from keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(150,150),
        batch_size=20,
        class_mode='categorical')

validation_generator = test_datagen.flow_from_directory(
        val_dir,
        target_size=(150,150),
        batch_size=20,
        class_mode='categorical')

Found 8185 images belonging to 120 classes.
Found 2037 images belonging to 120 classes.


In [14]:
train_generator.class_indices

{'affenpinscher': 0,
 'afghan_hound': 1,
 'african_hunting_dog': 2,
 'airedale': 3,
 'american_staffordshire_terrier': 4,
 'appenzeller': 5,
 'australian_terrier': 6,
 'basenji': 7,
 'basset': 8,
 'beagle': 9,
 'bedlington_terrier': 10,
 'bernese_mountain_dog': 11,
 'black-and-tan_coonhound': 12,
 'blenheim_spaniel': 13,
 'bloodhound': 14,
 'bluetick': 15,
 'border_collie': 16,
 'border_terrier': 17,
 'borzoi': 18,
 'boston_bull': 19,
 'bouvier_des_flandres': 20,
 'boxer': 21,
 'brabancon_griffon': 22,
 'briard': 23,
 'brittany_spaniel': 24,
 'bull_mastiff': 25,
 'cairn': 26,
 'cardigan': 27,
 'chesapeake_bay_retriever': 28,
 'chihuahua': 29,
 'chow': 30,
 'clumber': 31,
 'cocker_spaniel': 32,
 'collie': 33,
 'curly-coated_retriever': 34,
 'dandie_dinmont': 35,
 'dhole': 36,
 'dingo': 37,
 'doberman': 38,
 'english_foxhound': 39,
 'english_setter': 40,
 'english_springer': 41,
 'entlebucher': 42,
 'eskimo_dog': 43,
 'flat-coated_retriever': 44,
 'french_bulldog': 45,
 'german_shepherd'

In [25]:
from keras import layers
from keras import models
from keras import optimizers

model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu',
                        input_shape=(150, 150, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dense(512, activation='relu'))
model.add(layers.Dense(120, activation='sigmoid'))


model.compile(loss='categorical_crossentropy',
              optimizer=optimizers.RMSprop(lr=1e-4),
              metrics=['acc'])

history = model.fit_generator(
      train_generator,
      steps_per_epoch=100,
      epochs=30,
      validation_data=validation_generator,
      validation_steps=50)

Epoch 1/30
100/100 [==============================] - 155s 2s/step - loss: 4.8538 - acc: 0.0100 - val_loss: 4.7914 - val_acc: 0.0140
Epoch 2/30
100/100 [==============================] - 162s 2s/step - loss: 4.7980 - acc: 0.0125 - val_loss: 4.7899 - val_acc: 0.0140
Epoch 3/30
100/100 [==============================] - 155s 2s/step - loss: 4.7909 - acc: 0.0105 - val_loss: 4.7772 - val_acc: 0.0110
Epoch 4/30
100/100 [==============================] - 149s 1s/step - loss: 4.7825 - acc: 0.0115 - val_loss: 4.7633 - val_acc: 0.0120
Epoch 5/30
100/100 [==============================] - 145s 1s/step - loss: 4.7488 - acc: 0.0165 - val_loss: 4.7125 - val_acc: 0.0251
Epoch 6/30
100/100 [==============================] - 143s 1s/step - loss: 4.7040 - acc: 0.0185 - val_loss: 4.6677 - val_acc: 0.0180
Epoch 7/30
100/100 [==============================] - 143s 1s/step - loss: 4.6631 - acc: 0.0160 - val_loss: 4.6810 - val_acc: 0.0181
Epoch 8/30
100/100 [==============================] - 142s 1s/step - 

In [27]:
model.save('partly_trained.h5')

## Optional: Build a Baseline CNN

This is an optional step. Adapting a pretrained model will produce better results, but it may be interesting to create a CNN from scratch as a baseline. If you wish to, do so here.

In [ ]:
#Create a baseline CNN model

## Loading a Pretrained CNN

## Feature Engineering with the Pretrained Model

Now that you've loaded a pretrained model, it's time to adapt that convolutional base and add some fully connected layers on top in order to build a classifier from these feature maps.

In [5]:
#Your code here; add fully connected layers on top of the convolutional base
from keras.applications import VGG19
from keras import models, layers, optimizers
cnn_base = VGG19(weights='imagenet',
                 include_top=False,
                 input_shape=(240, 240, 3))

#Define Model Architecture
model = models.Sequential()
model.add(cnn_base)
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(120, activation='sigmoid'))

cnn_base.trainable = False

model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
vgg19 (Model)                (None, 7, 7, 512)         20024384  
_________________________________________________________________
flatten_2 (Flatten)          (None, 25088)             0         
_________________________________________________________________
dense_6 (Dense)              (None, 64)                1605696   
_________________________________________________________________
dense_7 (Dense)              (None, 128)               8320      
_________________________________________________________________
dense_8 (Dense)              (None, 256)               33024     
_________________________________________________________________
dense_9 (Dense)              (None, 128)               32896     
_________________________________________________________________
dense_10 (Dense)             (None, 120)               15480     
Total para

In [ ]:
#Compilation
import datetime

start = datetime.datetime.now()
model.compile(loss='categorical_crossentropy',
              optimizer=optimizers.RMSprop(lr=2e-5),
              metrics=['acc'])


#Fitting the Model
history = model.fit_generator(
              train_generator,
              steps_per_epoch= 20,
              epochs = 10,
              validation_data = validation_generator,
              validation_steps = 20)

end = datetime.datetime.now()
elapsed = end - start
print('{}'.format(elapsed))

Epoch 1/10
19/20 [===========================>..] - ETA: 10s - loss: 4.7953 - acc: 0.0105

## Visualize History

Now fit the model and visualize the training and validation accuracy/loss functions over successive epochs.

In [ ]:
#Your code here; visualize the training / validation history associated with fitting the model.

In [ ]:
#Save model

## Final Model Evaluation

In [ ]:
#Your code here

## Summary

Congratulations! In this lab, you brought all of your prior deep learning skills together from preprocessing including one-hot encoding, to adapting a pretrained model. There are always ongoing advancements in CNN architectures and best practices, but you have a solid foundation and understanding at this point.